# 🚀 Grad-Refactor-AI: Qwen2.5-Coder-3B-Instruct Native FP16 LoRA Training

**Hardware**: Nvidia Tesla T4 (Single GPU on Kaggle)  
**Architecture**: Native FP16 LoRA (All 7 linear projection layers) — **Zero bitsandbytes dependency to prevent shard-loading deadlocks**  
**Task**: Python Guard Clauses Refactoring (Prompt `refactor-v2`)  
**Expected Peak VRAM**: ~9.8 GB / 15.0 GB (fits comfortably with ~5 GB headroom)

In [ ]:
# Step 1: Environment Setup & Conflict Cleanup
import os, sys, subprocess

# 1. Force single GPU to prevent Kaggle Dual-T4 P2P deadlock
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. Remove conflicting torchao pre-installed in Kaggle images
!pip uninstall -y torchao
!pip install -q "transformers>=4.45.0" "peft>=0.13.0" "accelerate>=0.34.0" "datasets>=3.0.0"

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Extract Project Code & Dataset
import shutil, zipfile
from pathlib import Path

PROJECT = Path("/kaggle/working/grad-refactor-ai")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True, exist_ok=True)

# Locate dataset / code bundle from Kaggle input
input_zips = list(Path("/kaggle/input").rglob("*.zip"))
input_jsonl = list(Path("/kaggle/input").rglob("records.jsonl"))

if input_zips:
    print(f"Extracting bundle: {input_zips[0]}")
    with zipfile.ZipFile(input_zips[0]) as zf:
        zf.extractall(PROJECT)
elif input_jsonl:
    src_root = input_jsonl[0].parents[4] if len(input_jsonl[0].parents) >= 5 else input_jsonl[0].parent
    print(f"Copying files from: {src_root}")
    shutil.copytree(src_root, PROJECT, dirs_exist_ok=True)
else:
    print("Note: Upload your grad-refactor-ai-bundle as a Kaggle dataset or clone your repo here.")

print("Project contents:", sorted(p.name for p in PROJECT.iterdir()) if PROJECT.exists() else "empty")

In [ ]:
# Step 3: Load Qwen2.5-Coder-3B-Instruct in Native FP16 (Loads in ~10 seconds!)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

print(f"Downloading and loading {MODEL_ID} in native torch.float16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Native FP16 on GPU 0 - No bitsandbytes, zero deadlocks
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# Setup LoRA covering all 7 linear projection layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print(f"Allocated VRAM after load: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# Step 4: Tokenize Guard Clauses Dataset (Prompt refactor-v2 format)
import json
from datasets import Dataset

records_file = PROJECT / "datasets/guard_clauses/v1/records.jsonl"
if not records_file.exists():
    # Fallback search if path differs
    found = list(PROJECT.rglob("records.jsonl"))
    records_file = found[0] if found else None

assert records_file and records_file.exists(), "records.jsonl not found!"

train_rows = []
val_rows = []
with open(records_file, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if row["split"] == "train":
            train_rows.append(row)
        elif row["split"] == "validation":
            val_rows.append(row)

print(f"Loaded {len(train_rows)} train records and {len(val_rows)} validation records.")

def encode_record(row):
    # Build chat messages matching refactor-v2
    prompt_text = row["prompt"]
    completion_text = row["completion"] + "<|im_end|>"
    full_text = f"<|im_start|>system\nYou are a precise Python refactoring assistant.<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n{completion_text}"
    
    prompt_ids = tokenizer.encode(f"<|im_start|>system\nYou are a precise Python refactoring assistant.<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n", add_special_tokens=False)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    
    return {
        "input_ids": full_ids[:2048],
        "attention_mask": [1] * len(full_ids[:2048]),
        "labels": labels[:2048]
    }

train_dataset = Dataset.from_list([encode_record(r) for r in train_rows])
val_dataset = Dataset.from_list([encode_record(r) for r in val_rows])
print("Datasets successfully tokenized!")

In [ ]:
# Step 5: Train LoRA with AdamW
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = Path("/kaggle/working/runs/qwen3b-lora")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_steps=12,
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
)

print("Starting training...")
train_result = trainer.train()
print("Training complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

In [ ]:
# Step 6: Save Adapter & Package for Download
FINAL_ADAPTER = Path("/kaggle/working/qwen3b-adapter")
model.save_pretrained(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)

zip_path = shutil.make_archive("/kaggle/working/qwen3b-lora-adapter", "zip", FINAL_ADAPTER)
print(f"✅ Successfully saved and zipped adapter: {zip_path}")
print(f"Archive size: {Path(zip_path).stat().st_size / 1e6:.2f} MB")
print("You can now download 'qwen3b-lora-adapter.zip' from Kaggle's Output tab!")